# 05. Final Load Prep (Tableau Exports)
**Project:** Logistics & Delivery Delays Analysis (Olist)
**Focus:** Generating specific CSV extracts mapped exactly to Dashboards D1, D2, D3, and D4 of the Project Lead's requirements.


In [13]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PATH_CLEANED = "../data/processed/olist_cleaned_data.csv"
PATH_TABLEAU = "../data/processed/tableau_ready/"
os.makedirs(PATH_TABLEAU, exist_ok=True)

df = pd.read_csv(PATH_CLEANED)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q').dt.strftime('Q%q-%Y')
df['is_bad_review'] = df['review_score'] == 1
print(f"Base dataset loaded. Shape: {df.shape}")

def clean_colnames(dataframe):
    df_out = dataframe.copy()
    df_out.columns = [col.replace('_', ' ').title() for col in df_out.columns]
    return df_out


Base dataset loaded. Shape: (109657, 51)


## 1. D1 — Executive Pulse
**Level:** Time-series (Quarterly and Yearly)
**Metrics:** Total orders, late rate, avg delay, avg ETA difference


In [14]:
# Executive Pulse Extract
d1_extract = df.groupby(['order_year', 'order_quarter']).agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_delay_days=('delivery_delay', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

d1_extract['late_rate_pct'] = (d1_extract['late_orders'] / d1_extract['total_orders'] * 100).round(2)
d1_extract['avg_delay_days'] = d1_extract['avg_delay_days'].round(2)
d1_extract['avg_review_score'] = d1_extract['avg_review_score'].round(2)

out_d1 = os.path.join(PATH_TABLEAU, '01_executive_pulse.csv')
clean_colnames(d1_extract).to_csv(out_d1, index=False)
print(f"Exported: {out_d1} | Shape: {d1_extract.shape}")


Exported: ../data/processed/tableau_ready/01_executive_pulse.csv | Shape: (8, 7)


## 2. D2 — Geographic Risk Monitor
**Level:** Geographic Aggregations (State and Top Routes)
**Metrics:** Avg delay, late rate by seller state and shipping route


In [15]:
# State-level
d2_state = df.groupby('seller_state').agg(
    total_orders=('order_id', 'count'),
    avg_delay_days=('delivery_delay', 'mean'),
    late_rate_pct=('is_late', lambda x: x.mean() * 100)
).reset_index().round(2)

# Route-level (Seller State -> Customer State)
df['shipping_route'] = df['seller_state'] + ' -> ' + df['customer_state']
d2_route = df.groupby(['shipping_route', 'seller_state', 'customer_state']).agg(
    total_orders=('order_id', 'count'),
    avg_delay_days=('delivery_delay', 'mean'),
    late_rate_pct=('is_late', lambda x: x.mean() * 100)
).reset_index().round(2)

# Save both
out_d2_state = os.path.join(PATH_TABLEAU, '02a_geo_risk_state.csv')
out_d2_route = os.path.join(PATH_TABLEAU, '02b_geo_risk_route.csv')
clean_colnames(d2_state).to_csv(out_d2_state, index=False)
clean_colnames(d2_route).to_csv(out_d2_route, index=False)
print(f"Exported D2 State: Shape {d2_state.shape}")
print(f"Exported D2 Route: Shape {d2_route.shape}")


Exported D2 State: Shape (22, 4)
Exported D2 Route: Shape (409, 6)


## 3. D3 — Seller Performance Scorecard
**Level:** Individual Seller Accountability
**Guardrail:** Requires minimum 10 orders to ensure statistical validity


In [16]:
MIN_ORDERS_D3 = 10
d3_seller = df.groupby('seller_id').agg(
    total_orders=('order_id', 'count'),
    seller_state=('seller_state', 'first'),
    seller_city=('seller_city', 'first'),
    late_orders=('is_late', 'sum'),
    avg_delay_days=('delivery_delay', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

d3_seller = d3_seller[d3_seller['total_orders'] >= MIN_ORDERS_D3].copy()
d3_seller['delay_rate_pct'] = (d3_seller['late_orders'] / d3_seller['total_orders'] * 100).round(2)
d3_seller['avg_delay_days'] = d3_seller['avg_delay_days'].round(2)
d3_seller['avg_review_score'] = d3_seller['avg_review_score'].round(2)

# Flag sellers with >13% delay rate
d3_seller['high_risk_seller'] = (d3_seller['delay_rate_pct'] > 13).map({True: 'Yes', False: 'No'})

out_d3 = os.path.join(PATH_TABLEAU, '03_seller_scorecard.csv')
clean_colnames(d3_seller).to_csv(out_d3, index=False)
print(f"Exported: {out_d3} | Shape: {d3_seller.shape}")


Exported: ../data/processed/tableau_ready/03_seller_scorecard.csv | Shape: (1338, 9)


## 4. D4 — Customer Experience Impact
**Level:** Granular (Sampled down for Tableau scatter plotting limits)
**Metrics:** Delivery delay vs review score scattered mapping


In [17]:
# Extract a 10% representative sample of order-level data for the scatter plot
d4_customer = df[['order_id', 'order_purchase_timestamp', 'delivery_delay', 'review_score', 'is_late', 'is_bad_review', 'seller_state', 'customer_state']].copy()
d4_customer['is_late'] = d4_customer['is_late'].map({True: 'Late', False: 'On Time'})
d4_customer['is_bad_review'] = d4_customer['is_bad_review'].map({True: '1-Star', False: '2-5 Star'})

# Sample to 25,000 rows so Tableau doesn't freeze on scatter plots
d4_sample = d4_customer.sample(n=min(25000, len(d4_customer)), random_state=42)

out_d4 = os.path.join(PATH_TABLEAU, '04_customer_impact.csv')
clean_colnames(d4_sample).to_csv(out_d4, index=False)
print(f"Exported: {out_d4} | Shape: {d4_sample.shape}")
print("Final Load processing complete.")


Exported: ../data/processed/tableau_ready/04_customer_impact.csv | Shape: (25000, 8)
Final Load processing complete.
